In [13]:
# Ollama 기반 CSV 챗봇 예제에 필요한 라이브러리를 불러옵니다.
import subprocess

from llama_index.readers.file import PDFReader
from llama_index.llms.ollama import Ollama
from llama_index.embeddings.ollama import OllamaEmbedding
from llama_index.core import Settings, VectorStoreIndex, Document
from pathlib import Path
import pandas as pd


In [14]:
# CSV 경로와 로컬 Ollama LLM/임베딩 모델을 설정합니다.
CSV_PATH = Path('../Data/ChatbotData.csv')
ollama_base_url = 'http://localhost:11434'
ollama_llm_model = 'gemma3n:e4b'
ollama_embed_model = 'nomic-embed-text'

def installed_ollama_models() -> set[str]:
    try:
        result = subprocess.run(
            ['ollama', 'list'],
            check=True,
            capture_output=True,
            text=True,
        )
    except FileNotFoundError as exc:
        raise RuntimeError('Ollama CLI가 설치되어 있지 않습니다. https://ollama.com 에서 설치하세요.') from exc
    except subprocess.CalledProcessError as exc:
        raise RuntimeError('Ollama 서버가 실행 중인지 확인하세요. 터미널에서 `ollama serve`를 실행하세요.') from exc

    names = set()
    for line in result.stdout.splitlines()[1:]:
        parts = line.split()
        if parts:
            names.add(parts[0])
    return names

def ensure_ollama_model(model_name: str) -> None:
    installed = installed_ollama_models()
    candidates = {model_name}
    if ':' not in model_name:
        candidates.add(f'{model_name}:latest')

    if installed.intersection(candidates):
        print(f'이미 설치됨: {model_name}')
        return

    print(f'Ollama 모델 다운로드 중: {model_name}')
    subprocess.run(['ollama', 'pull', model_name], check=True)

for model_name in [ollama_llm_model, ollama_embed_model]:
    ensure_ollama_model(model_name)

Settings.llm = Ollama(
    model=ollama_llm_model,
    base_url=ollama_base_url,
    request_timeout=120.0,
    temperature = 0 # 낮을수록 좋은거. gpt 기준 0.7이 일반, 0.5가 plus, 0.2가 pro 정도 된다고 함  

)
Settings.embed_model = OllamaEmbedding(
    model_name=ollama_embed_model,
    base_url=ollama_base_url,
    request_timeout=120.0,
    embed_batch_size=64,
)

print(f"CSV경로: {CSV_PATH.resolve()}")
print(f"Ollama / LlamaIndex 설정 완료")

이미 설치됨: gemma3n:e4b
이미 설치됨: nomic-embed-text
CSV경로: /Users/cheng80/Documents/WorkSpace/RAG/Data/ChatbotData.csv
Ollama / LlamaIndex 설정 완료


#### CSV를 문서 형태로 변환

In [15]:
# 챗봇 데이터 CSV를 DataFrame으로 읽습니다.
df = pd.read_csv(CSV_PATH)
df.head()

,Q,A,label
0,12시 땡!,하루가 또 가네요.,0
1,1지망 학교 떨어졌어,위로해 드립니다.,0
2,3박4일 놀러가고 싶다,여행은 언제나 좋죠.,0
3,3박4일 정도 놀러가고 싶다,여행은 언제나 좋죠.,0
4,PPL 심하네,눈살이 찌푸려지죠.,0


In [16]:
# 벡터화할 텍스트 컬럼과 metadata 컬럼을 지정합니다.
TEXT_COLUMNS = ['Q','A']
METADATA_COLUMNS = ['label']

# MAX_ROWS = 1000
# df = df.head(MAX_ROWS).copy()

display(df.head())
print('문서와 대상 컬럼 : ',TEXT_COLUMNS)
print('메타데이터 컬럼 : ',METADATA_COLUMNS)
# print('사용할 행수 : ',MAX_ROWS)


,Q,A,label
0,12시 땡!,하루가 또 가네요.,0
1,1지망 학교 떨어졌어,위로해 드립니다.,0
2,3박4일 놀러가고 싶다,여행은 언제나 좋죠.,0
3,3박4일 정도 놀러가고 싶다,여행은 언제나 좋죠.,0
4,PPL 심하네,눈살이 찌푸려지죠.,0


문서와 대상 컬럼 :  ['Q', 'A']
메타데이터 컬럼 :  ['label']


In [17]:
# CSV의 각 행을 LlamaIndex Document 객체로 변환합니다.
# 각 Row를 질문-답변 형태의 문서로 변환

def row_to_document(row:pd.Series, row_number:int) -> Document:
    text_parts = []

    for column in TEXT_COLUMNS:
        value = row[column]

        if pd.isna(value):
            continue
        text_parts.append(f'{column}:{value}')

    metadata = {
        'row_number' : row_number,
        'label' : row['label']
    }
    return Document(
        text = ' | '.join(text_parts),
        metadata = metadata
    )    
# DataFrame의 각 row를 Document로 변환
documents = [row_to_document(row, idx) for idx, row in df.iterrows()]

print('생성된 Document수 : ',len(documents))
print('첫번째 Document 예제')
print(documents[0].text)

생성된 Document수 :  11823
첫번째 Document 예제
Q:12시 땡! | A:하루가 또 가네요.


In [18]:
# 변환한 문서들로 벡터 인덱스와 chat engine을 생성합니다.
# 문서목록으로 벡터 인덱스 생성
index = VectorStoreIndex.from_documents(documents, show_progress=True)

# 검색된 문서를 바탕으로 답변하는 chat engine을 제작
# as_query_engine는 검색하여 답변
# as_chat_engine은 추론
chat_engine = index.as_chat_engine(
    chat_mode='context',
    similarity_top_k = 5,
    verbose = True
)

/Users/cheng80/Documents/WorkSpace/RAG/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Generating embeddings: 100%|██████████| 1583/1583 [00:40<00:00, 39.45it/s]
2026-06-02 11:55:22,828 - INFO - HTTP Request: POST http://localhost:11434/api/show "HTTP/1.1 200 OK"


In [19]:
# 고정 질문으로 chat engine 동작을 테스트합니다.
# Test
question = '12시 땡! 이라는 질문에는 어떤 답변이 연결되어 있어?'
response = chat_engine.chat(question)
print('질문:',question)
print('응답:',response)

2026-06-02 11:55:22,897 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-06-02 11:55:36,644 - INFO - HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"


질문: 12시 땡! 이라는 질문에는 어떤 답변이 연결되어 있어?
응답: 12시 땡! 이라는 질문에는 "하루가 또 가네요." 라는 답변이 연결되어 있습니다.


In [20]:
# 사용자가 exit 또는 quit를 입력할 때까지 질문을 반복해서 받습니다.
# 여러번 질문하고
# exit, quit 종료
while True:
    user_question = input('질문을 입력하세요:').strip()
    if user_question.lower() in {'exit', 'quit'}:
        print('쳇봇을 종료합니다.')
        break
    if not user_question:
        print('빈 질문은 처리 할 수 없다. 다시 입력하여라')
        continue

    answer = chat_engine.chat(user_question)
    print('\n[응답]')
    print(answer)
    print('-'*50)    


2026-06-02 11:58:58,535 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-06-02 11:59:08,185 - INFO - HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"



[응답]
죄송해요. 그럴 수도 있겠네요.
--------------------------------------------------
쳇봇을 종료합니다.
